In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from utils import ChocolateCountDataset, ChocolateCounterNetDeep
from tqdm import tqdm
import os

from sklearn.model_selection import train_test_split


In [2]:
import os

all_filenames = [f for f in os.listdir("../dataset_project_iapr2025/train") if f.endswith(('.JPG'))]

train_filenames, val_filenames = train_test_split(all_filenames[13:], test_size=0.33, random_state=42)

train_filenames += all_filenames[:13]

In [ ]:

# Paths (adjust as needed)
TRAIN_IMAGE_DIR = "../dataset_project_iapr2025/train"
VAL_IMAGE_DIR = "../dataset_project_iapr2025/train"
LABEL_DIR = "./labels"

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "best_model.pth"

# Dataset and loaders
train_dataset = ChocolateCountDataset(
    image_filenames=train_filenames,
    image_dir=TRAIN_IMAGE_DIR,
    label_dir=LABEL_DIR,
    augment=True,
    n_augmentations=3
)
val_dataset = ChocolateCountDataset(
    image_filenames=val_filenames,
    image_dir=VAL_IMAGE_DIR,
    label_dir=LABEL_DIR,
    augment=False
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Model, loss, optimizer
model = ChocolateCounterNetDeep().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop
def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0
    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images, targets = images.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    avg_train_loss = running_loss / len(train_loader.dataset)
    val_loss = evaluate(val_loader)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Model saved to {SAVE_PATH}")

# Csv file generation

# Functions training and evaluation :

# Check the transformations :